# Simulation result explorer

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import scienceplots  # noqa: F401
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib_inline.backend_inline import set_matplotlib_formats

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
extra = 3
plt.rcParams.update({
    "font.size": 14+extra,
    "axes.titlesize": 17+extra,
    "axes.labelsize": 15+extra,
    "xtick.labelsize": 12+extra,
    "ytick.labelsize": 12+extra,
    "legend.fontsize": 12+extra,
    "legend.title_fontsize": 13+extra,
    "figure.titlesize": 19+extra,
})

project_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "tasks").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from the network-creation repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.plotting.plot_pair import plot_2d_descriptor, plot_pair
from network_creation.plotting.reference_values import get_observer_reference_style
from tasks.policies import load_policy_results

In [ ]:
extra = 8
CONFIG = {
    "result_store": (
        project_root
        / "artifacts"
        / "simulation_runs"
        / "run-20260901T230504.zarr"
    ),
    "normalize_time": True,
    "export": {
        "output_root": project_root / "artifacts" / "figures",
        "dpi": 300,
    },
    "observables": {
        "centered_mean_welfare.mean": {
            "label": "Terminal star-normalized welfare",
            "limits": (0.0, 1.0),
        },
        "hub_sponsorship.mean": {
            "label": "Mean terminal hub sponsorship",
            "limits": (0.0, 1.0),
        },
        "centered_mean_welfare.std": {
            "label": "SD of terminal welfare",
            "limits": (0.0, None),
            "scale_per_n_agents": False,  # False uses one global scale.
        },
    },
    "plot": {
        "method": "gaussian",
        "cmap": "jet",
        "resolution": 50,
        "point_size": 4,
        "pair_atol": 1e-5,
        "figure_side": 4,
        "subplot_spacing": 0.06,
        "visibility_grid_panel_side": 2.35,
        "visibility_grid_row_label_width": 0.5,
        "visibility_grid_strip_height": 0.38,
        "visibility_grid_spacing": 0.04,  # Shared row/column gap.
        "visibility_grid_font_sizes": {
            "axis_ticks": 15+extra,
            "axis_labels": 18+extra,
            "column_headers": 17+extra,
            "row_headers": 17+extra,
            "colorbar_ticks": 15+extra,
            "colorbar_label": 18+extra,
            "table_body": 15+extra,
            "table_header": 15+extra,
        },
        "subplot_margins": {
            "left": 0.125,
            "right": 0.895,
            "bottom": 0.11,
            "top": 0.88,
        },
        "colorbar_bounds": (0.2, 0.0, 0.08, 1.0),
        "single_colorbar_bounds": (1.08, 0.0, 0.05, 1.0),
        "reference_label_spacing": 0.07,
        "parameter_labels": {
            "visibility_distance": r"$\rho$",
            "exploration": r"$\tau$",
            "outside_optimism": r"$\omega$",
        },
        "reference_labels": {
            "star": "Efficient star",
            "random": "Max ER",
            "choi2020": "Choi_2020",
            "empty": r"$\leq$ Empty network",
            "Center_sponsored_star": "CSS",
            "Periphery_sponsored_star": "PSS",
            "Empty/Balanced": "Balanced",
        },
        "show_choi2020_reference": True,
    },
}

In [ ]:
results = load_policy_results(
    CONFIG["result_store"],
    stats=tuple(CONFIG["observables"]),
    normalize_time=CONFIG["normalize_time"],
)
last_step = results["x"].isel(step=-1, drop=True).rename("last_step")

composition_dims = tuple(
    dim
    for dim in results["x"].dims
    if dim not in {"n_agents", "parameter_row", "step", "observable"}
)
if len(composition_dims) != 1:
    raise ValueError(
        f"Expected one policy-composition dimension, found {composition_dims}."
    )
composition_dim = composition_dims[0]

loaded_path = Path(results.attrs["result_store_path"])
print(f"Loaded {loaded_path.relative_to(project_root)}")
print(f"Has failures: {results.attrs['has_failures']}")
print(f"Last shared step: {float(results.coords['step'][-1]):g}")
print(f"Time axis: {results.attrs.get('time_axis', 'raw_simulator_steps')}")

In [ ]:
def resolve_norm(observable, values, references):
    finite = np.asarray(values, dtype=float).ravel()
    finite = finite[np.isfinite(finite)]
    if references:
        finite = np.concatenate([finite, np.asarray(list(references.values()))])
    if not finite.size:
        raise ValueError(f"No finite values are available for {observable!r}.")

    configured_min, configured_max = CONFIG["observables"][observable]["limits"]
    vmin = float(finite.min()) if configured_min is None else float(configured_min)
    vmax = float(finite.max()) if configured_max is None else float(configured_max)
    return Normalize(vmin=vmin, vmax=vmax)


def spread_reference_label_positions(positions, lower, upper, min_spacing):
    positions = np.asarray(positions, dtype=float).copy()
    if positions.size < 2:
        return positions

    min_spacing = min(
        float(min_spacing), (upper - lower) / (positions.size - 1)
    )
    for _ in range(256):
        previous = positions.copy()
        for index in range(positions.size - 1):
            overlap = min_spacing - (positions[index + 1] - positions[index])
            if overlap > 0.0:
                positions[index] -= overlap / 2.0
                positions[index + 1] += overlap / 2.0
        positions = np.clip(positions, lower, upper)
        if np.max(np.abs(positions - previous)) < 1e-9:
            break
    return positions


def group_close_reference_positions(positions, threshold):
    if not positions:
        return []

    groups = [[0]]
    for index in range(1, len(positions)):
        if positions[index] - positions[index - 1] < threshold:
            groups[-1].append(index)
        else:
            groups.append([index])
    return groups


def draw_color_scale(
    axis,
    norm,
    cmap,
    references,
    label,
    has_under_range,
    reference_label_spacing=None,
):
    colorbar = axis.figure.colorbar(
        ScalarMappable(norm=norm, cmap=cmap),
        cax=axis,
        extend="min" if has_under_range else "neither",
    )

    visible_references = {
        name: value
        for name, value in references.items()
        if name != "choi2020"
        or CONFIG["plot"]["show_choi2020_reference"]
    }
    if visible_references:
        items = sorted(visible_references.items(), key=lambda item: item[1])
        tick_values = [value for _, value in items]
        tick_labels = [
            f"{CONFIG['plot']['reference_labels'].get(name, name)} ({value:.2f})"
            for name, value in items
        ]
        tick_positions = [float(norm(value)) for value in tick_values]
        if reference_label_spacing is None:
            reference_label_spacing = CONFIG["plot"][
                "reference_label_spacing"
            ]
        label_positions = spread_reference_label_positions(
            tick_positions,
            0.0,
            1.0,
            reference_label_spacing,
        )
        label_groups = group_close_reference_positions(
            tick_positions, reference_label_spacing
        )
    else:
        tick_values = MaxNLocator(nbins=6).tick_values(norm.vmin, norm.vmax)
        tick_values = [
            value for value in tick_values if norm.vmin <= value <= norm.vmax
        ]
        tick_labels = [f"{value:.2f}" for value in tick_values]
        tick_positions = [float(norm(value)) for value in tick_values]

    axis.set_yticks(
        tick_values,
        labels=[""] * len(tick_positions) if visible_references else tick_labels,
    )
    axis.minorticks_off()
    axis.yaxis.tick_right()
    axis.yaxis.set_label_position("left")
    colorbar.set_label(label)
    if visible_references:
        label_transform = axis.transAxes
        rail_x = 1.22
        line_end_x = 1.32
        text_x = 1.45
        line_style = {
            "color": "black",
            "linewidth": 0.8,
            "solid_capstyle": "butt",
            "clip_on": False,
            "transform": label_transform,
        }
        for group in label_groups:
            group_ticks = [tick_positions[index] for index in group]
            group_labels = [label_positions[index] for index in group]
            for tick_position in group_ticks:
                axis.plot([1.0, rail_x], [tick_position, tick_position], **line_style)
            axis.plot(
                [rail_x, rail_x],
                [min(*group_ticks, *group_labels), max(*group_ticks, *group_labels)],
                **line_style,
            )
            for label_position in group_labels:
                axis.plot(
                    [rail_x, line_end_x],
                    [label_position, label_position],
                    **line_style,
                )
        for label_position, tick_label in zip(
            label_positions, tick_labels, strict=True
        ):
            axis.text(
                text_x,
                label_position,
                tick_label,
                transform=label_transform,
                ha="left",
                va="center",
                fontsize=plt.rcParams["ytick.labelsize"],
                clip_on=False,
            )
    colorbar.outline.set_edgecolor("black")
    colorbar.outline.set_linewidth(1.0)


def plot_last_step(observable, n_agents, composition):
    selector = {
        "observable": observable,
        "n_agents": n_agents,
        composition_dim: composition,
    }
    values = last_step.sel(selector)
    theta = results["theta"].sel(
        n_agents=n_agents,
        **{composition_dim: composition},
    )
    valid = values.notnull() & theta.notnull().all("theta_parameter")
    values = values.where(valid, drop=True)
    theta = theta.where(valid, drop=True)
    if not values.sizes.get("parameter_row", 0):
        raise ValueError("No finite final values match this selection.")

    plot_config = CONFIG["plot"]
    varying_parameters = int(
        (theta.var("parameter_row") > plot_config["pair_atol"]).sum().item()
    )
    n_axes = varying_parameters - 1
    if n_axes < 1:
        raise ValueError("At least two sampled parameters must vary for a pair plot.")

    theta_names = [str(name) for name in theta.coords["theta_parameter"].values]
    parameter_labels = plot_config["parameter_labels"]
    labels = [parameter_labels.get(name, name) for name in theta_names]

    references = {}
    if observable.split(".")[1:2] == ["mean"]:
        references, _ = get_observer_reference_style(
            observable,
            values.to_numpy(),
            n_agents=int(n_agents),
            theta_labels=theta_names,
            theta_values=theta.to_numpy(),
        )

    scale_selector = {"observable": observable}
    if CONFIG["observables"][observable].get("scale_per_n_agents", False):
        scale_selector["n_agents"] = n_agents
    shared_values = last_step.sel(scale_selector).to_numpy()
    norm = resolve_norm(observable, shared_values, references)
    base_cmap = plt.get_cmap(plot_config["cmap"])
    cmap = base_cmap.with_extremes(under=base_cmap(0.0))
    has_under_range = bool(
        norm.vmin is not None
        and np.any(np.isfinite(shared_values) & (shared_values < norm.vmin))
    )

    with plt.ioff():
        fig, axes = plt.subplots(
            n_axes,
            n_axes,
            squeeze=False,
            figsize=(plot_config["figure_side"] * n_axes,) * 2,
            gridspec_kw={
                "wspace": plot_config["subplot_spacing"],
                "hspace": plot_config["subplot_spacing"],
            },
        )
        fig.subplots_adjust(**plot_config["subplot_margins"])
    plot_pair(
        X=theta.to_numpy(),
        y=values.to_numpy(),
        labels=labels,
        ordered_discrete_keys=[
            parameter_labels.get("visibility_distance", "visibility_distance")
        ]
        if "visibility_distance" in theta_names
        else [],
        atol=plot_config["pair_atol"],
        cmap=cmap,
        method=plot_config["method"],
        resolution=plot_config["resolution"],
        point_size=plot_config["point_size"],
        norm=norm,
        fig=fig,
        axes=axes,
    )

    if n_axes == 1:
        colorbar_axis = axes[0, 0].inset_axes(
            plot_config["single_colorbar_bounds"]
        )
    else:
        colorbar_slot = axes[0, -1]
        colorbar_slot.clear()
        colorbar_slot.axis("off")
        colorbar_axis = colorbar_slot.inset_axes(plot_config["colorbar_bounds"])
    draw_color_scale(
        colorbar_axis,
        norm,
        cmap,
        references,
        CONFIG["observables"][observable]["label"],
        has_under_range,
    )
    fig.align_ylabels(axes[:, 0])

    composition_label = composition_dim.replace("_", " ")
    """
    fig.suptitle(
        f"{CONFIG['observables'][observable]['label']}\n"
        f"n_agents={int(n_agents)}, {composition_label}={composition:g}"
    )
    """
    return fig


def build_visibility_grid_colorbar_and_table(
    fig,
    observable,
    cmap,
    row_styles,
    n_agents_values,
    row_strips,
    strip_edgecolor,
):
    plot_config = CONFIG["plot"]
    font_sizes = plot_config["visibility_grid_font_sizes"]
    shared_norm = next(iter(row_styles.values()))[0]
    has_shared_under_range = any(style[2] for style in row_styles.values())
    observer_name, aggregation, *_ = observable.split(".")
    is_welfare_mean = (
        observer_name == "centered_mean_welfare" and aggregation == "mean"
    )
    scale_bounds = (
        (0.1, 0.10, 0.27, 0.025)
        if is_welfare_mean
        else (0.25, 0.10, 0.50, 0.025)
    )
    scale_axis = fig.add_axes(scale_bounds)
    colorbar = fig.colorbar(
        ScalarMappable(norm=shared_norm, cmap=cmap),
        cax=scale_axis,
        orientation="horizontal",
        extend="min" if has_shared_under_range else "neither",
    )

    reference_rows = [row_styles[n_agents][1] for n_agents in n_agents_values]
    shared_references = reference_rows[0]
    if is_welfare_mean:
        colorbar.set_ticks(
            [
                shared_references.get("empty", shared_norm.vmin),
                shared_references.get("star", shared_norm.vmax),
            ],
            labels=[r"$\leq$ Empty network", "Efficient star"],
        )
    else:
        references_are_shared = bool(shared_references) and all(
            references.keys() == shared_references.keys()
            and all(
                np.isclose(references[name], value)
                for name, value in shared_references.items()
            )
            for references in reference_rows[1:]
        )
        if references_are_shared:
            tick_values = []
            tick_labels = []
            visible_references = [
                (name, value)
                for name, value in shared_references.items()
                if name != "choi2020"
                or plot_config["show_choi2020_reference"]
            ]
            for name, value in sorted(
                visible_references, key=lambda item: item[1]
            ):
                label = (
                    "Empty network"
                    if name == "empty"
                    else plot_config["reference_labels"].get(
                        name, name.replace("_", " "),
                    )
                )
                if tick_values and np.isclose(value, tick_values[-1]):
                    tick_labels[-1] += f" / {label}"
                else:
                    tick_values.append(value)
                    tick_labels.append(label)
            if tick_values:
                colorbar.set_ticks(tick_values, labels=tick_labels)

    colorbar.ax.tick_params(
        which="both", length=0, labelsize=font_sizes["colorbar_ticks"]
    )
    colorbar.outline.set_edgecolor("black")
    colorbar.outline.set_linewidth(1.0)
    colorbar.set_label(CONFIG["observables"][observable]["label"])
    colorbar.ax.xaxis.label.set_size(font_sizes["colorbar_label"])

    if not is_welfare_mean:
        return None

    table_width = 0.50
    table_bounds = (0.98 - table_width, 0.0, table_width, 0.17)
    n_column_width = row_strips[0].get_position().width / table_bounds[2]
    reference_column_width = (1.0 - n_column_width) / 4.0
    table_axis = fig.add_axes(table_bounds)
    table_axis.axis("off")
    table_rows = []
    table_colors = []
    for n_agents in n_agents_values:
        references = row_styles[n_agents][1]
        row_values = [
            references.get("empty"),
            references.get("random"),
            references.get("choi2020"),
            references.get("star"),
        ]
        table_rows.append(
            [
                *["--" if value is None else f"{value:.2f}" for value in row_values],
                f"{n_agents}",
            ]
        )
        table_colors.append(
            [
                *[
                    "white" if value is None else cmap(shared_norm(value))
                    for value in row_values
                ],
                "white",
            ]
        )
    reference_table = table_axis.table(
        cellText=table_rows,
        cellColours=table_colors,
        colLabels=["Empty\nnetwork", r"Max. $\overrightarrow{\mathrm{ER}}$", "Exp.\nmean", "Efficient\nstar", r"$n$"],
        colWidths=[reference_column_width] * 4 + [n_column_width],
        cellLoc="center",
        colLoc="center",
        bbox=[0.0, 0.0, 1.0, 1.0],
    )
    reference_table.auto_set_font_size(False)
    reference_table.set_fontsize(font_sizes["table_body"])
    for column in range(5):
        header_cell = reference_table[0, column]
        header_cell.set_height(2.0 * header_cell.get_height())
        header_cell.set_facecolor("white")
        header_cell.get_text().set_fontsize(font_sizes["table_header"])
    for row, n_agents in enumerate(n_agents_values, start=1):
        references = row_styles[n_agents][1]
        for column, key in enumerate(
            ("empty", "random", "choi2020", "star")
        ):
            value = references.get(key)
            if value is None:
                continue
            rgb = np.asarray(cmap(shared_norm(value))[:3])
            linear_rgb = np.where(
                rgb <= 0.04045,
                rgb / 12.92,
                ((rgb + 0.055) / 1.055) ** 2.4,
            )
            luminance = float(linear_rgb @ [0.2126, 0.7152, 0.0722])
            text_color = "white" if luminance < 0.179 else "black"
            reference_table[row, column].get_text().set_color(text_color)
    for cell in reference_table.get_celld().values():
        cell.set_edgecolor(strip_edgecolor)
        cell.set_linewidth(0.8)
    return reference_table


def plot_visibility_grid(observable, composition):
    parameter_names = [
        str(name) for name in results["theta"].coords["theta_parameter"].values
    ]
    required_parameters = {
        "visibility_distance",
        "outside_optimism",
        "exploration",
    }
    missing_parameters = required_parameters.difference(parameter_names)
    if missing_parameters:
        raise ValueError(
            "The visibility grid requires: "
            + ", ".join(sorted(missing_parameters))
        )

    selector = {composition_dim: composition}
    values = last_step.sel(observable=observable, **selector)
    theta = results["theta"].sel(selector)
    valid = np.isfinite(values) & np.isfinite(theta).all("theta_parameter")

    visibility_data = (
        theta.sel(theta_parameter="visibility_distance").where(valid).to_numpy()
    )
    visibilities = np.unique(visibility_data[np.isfinite(visibility_data)])
    if not visibilities.size:
        raise ValueError("No visibility values are available for this selection.")

    omega_data = (
        theta.sel(theta_parameter="outside_optimism").where(valid).to_numpy()
    )
    temperature_data = (
        theta.sel(theta_parameter="exploration").where(valid).to_numpy()
    )
    omega_data = omega_data[np.isfinite(omega_data)]
    temperature_data = temperature_data[np.isfinite(temperature_data)]
    if not omega_data.size or not temperature_data.size:
        raise ValueError("No finite omega-temperature pairs match this selection.")
    plot_config = CONFIG["plot"]
    font_sizes = plot_config["visibility_grid_font_sizes"]
    omega_limits = tuple(
        float(value)
        for value in MaxNLocator().tick_values(omega_data.min(), omega_data.max())[[0, -1]]
    )
    temperature_limits = tuple(
        float(value)
        for value in MaxNLocator().tick_values(
            temperature_data.min(), temperature_data.max()
        )[[0, -1]]
    )
    bounds = (*omega_limits, *temperature_limits)

    shared_values = values.where(valid).to_numpy()
    base_cmap = plt.get_cmap(plot_config["cmap"])
    cmap = base_cmap.with_extremes(under=base_cmap(0.0))
    n_agents_values = [int(value) for value in values.coords["n_agents"].values]
    row_styles = {}
    for n_agents in n_agents_values:
        row_values = values.sel(n_agents=n_agents).to_numpy()
        row_theta = theta.sel(n_agents=n_agents).to_numpy()
        row_valid = valid.sel(n_agents=n_agents).to_numpy()
        references = {}
        if observable.split(".")[1:2] == ["mean"]:
            references, _ = get_observer_reference_style(
                observable,
                row_values[row_valid],
                n_agents=n_agents,
                theta_labels=parameter_names,
                theta_values=row_theta[row_valid],
            )
        scale_values = (
            row_values[row_valid]
            if CONFIG["observables"][observable].get(
                "scale_per_n_agents", False
            )
            else shared_values
        )
        norm = resolve_norm(observable, scale_values, references)
        has_under_range = bool(
            norm.vmin is not None
            and np.any(np.isfinite(row_values) & (row_values < norm.vmin))
        )
        row_styles[n_agents] = (norm, references, has_under_range)

    panel_side = plot_config["visibility_grid_panel_side"]
    row_label_width = plot_config["visibility_grid_row_label_width"]
    strip_height = plot_config["visibility_grid_strip_height"]
    grid_spacing = plot_config["visibility_grid_spacing"]
    gap = grid_spacing * panel_side
    figure_width = (
        row_label_width
        + len(visibilities) * panel_side
        + len(visibilities) * gap
    )
    figure_height = (
        strip_height
        + len(n_agents_values) * panel_side
        + len(n_agents_values) * gap
        + 2.15
    )
    with plt.ioff():
        fig = plt.figure(figsize=(figure_width, figure_height))
        grid = fig.add_gridspec(
            len(n_agents_values) + 1,
            len(visibilities) + 1,
            width_ratios=(
                [1.0] * len(visibilities)
                + [row_label_width / panel_side]
            ),
            height_ratios=[strip_height / panel_side]
            + [1.0] * len(n_agents_values),
            left=0.07,
            right=0.98,
            bottom=0.25,
            top=0.97,
            wspace=grid_spacing,
            hspace=grid_spacing,
        )
        axes = np.empty((len(n_agents_values), len(visibilities)), dtype=object)
        for row in range(len(n_agents_values)):
            for column in range(len(visibilities)):
                axes[row, column] = fig.add_subplot(grid[row + 1, column])

    for row, n_agents in enumerate(n_agents_values):
        row_values = values.sel(n_agents=n_agents).to_numpy()
        row_theta = theta.sel(n_agents=n_agents)
        row_visibility = row_theta.sel(
            theta_parameter="visibility_distance"
        ).to_numpy()
        row_omega = row_theta.sel(
            theta_parameter="outside_optimism"
        ).to_numpy()
        row_temperature = row_theta.sel(
            theta_parameter="exploration"
        ).to_numpy()
        row_valid = valid.sel(n_agents=n_agents).to_numpy()
        norm, _, _ = row_styles[n_agents]

        for column, visibility in enumerate(visibilities):
            axis = axes[row, column]
            selected = row_valid & np.isclose(
                row_visibility,
                visibility,
                atol=plot_config["pair_atol"],
                rtol=0.0,
            )
            if not np.any(selected):
                axis.axis("off")
                continue

            plot_2d_descriptor(
                x=row_omega[selected],
                y=row_temperature[selected],
                z=row_values[selected],
                method=plot_config["method"],
                resolution=plot_config["resolution"],
                ax=axis,
                norm=norm,
                bounds=bounds,
                cmap=cmap,
                point_size=plot_config["point_size"],
            )
            axis.grid(False)
            omega_ticks = np.linspace(*omega_limits, 3)
            temperature_ticks = np.linspace(*temperature_limits, 3)
            axis.set_xticks(
                omega_ticks,
                labels=[
                    r"$1/2$" if np.isclose(value, 0.5) else f"{value:g}"
                    for value in omega_ticks
                ],
            )
            axis.set_yticks(
                temperature_ticks,
                labels=[f"{value:g}" for value in temperature_ticks],
            )
            axis.get_xticklabels()[0].set_ha("left")
            axis.get_xticklabels()[-1].set_ha("right")
            axis.get_yticklabels()[0].set_va("bottom")
            axis.get_yticklabels()[-1].set_va("top")
            axis.tick_params(axis="both", which="major", length=5, direction="inout")
            axis.minorticks_off()
            if row != len(n_agents_values) - 1:
                axis.tick_params(
                    axis="x",
                    which="both",
                    bottom=False,
                    top=False,
                    labelbottom=False,
                )
            if column != 0:
                axis.tick_params(
                    axis="y",
                    which="both",
                    left=False,
                    right=False,
                    labelleft=False,
                )

    # Reapply label visibility after all locators have created their ticks.
    for row in range(len(n_agents_values)):
        for column in range(len(visibilities)):
            axis = axes[row, column]
            if row != len(n_agents_values) - 1:
                for label in axis.get_xticklabels(which="both"):
                    label.set_visible(False)
            if column != 0:
                for label in axis.get_yticklabels(which="both"):
                    label.set_visible(False)

    strip_facecolor = "#f2f2f2"
    strip_edgecolor = "#b7b7b7"

    def style_strip(strip):
        strip.set_facecolor(strip_facecolor)
        strip.set_xticks([])
        strip.set_yticks([])
        for spine in strip.spines.values():
            spine.set_color(strip_edgecolor)
            spine.set_linewidth(0.8)

    for column, visibility in enumerate(visibilities):
        strip = fig.add_subplot(grid[0, column])
        style_strip(strip)
        strip.text(
            0.5,
            0.5,
            rf"$\rho = {visibility:g}$",
            ha="center",
            va="center",
            fontsize=font_sizes["column_headers"],
        )

    row_strips = []
    for row, n_agents in enumerate(n_agents_values):
        strip = fig.add_subplot(grid[row + 1, -1])
        row_strips.append(strip)
        style_strip(strip)
        strip.text(
            0.5,
            0.5,
            rf"$n={n_agents}$",
            ha="center",
            va="center",
            rotation=90,
            fontsize=font_sizes["row_headers"],
        )

    build_visibility_grid_colorbar_and_table(
        fig,
        observable,
        cmap,
        row_styles,
        n_agents_values,
        row_strips,
        strip_edgecolor,
    )

    for column in range(len(visibilities)):
        axes[-1, column].set_xlabel(
            r"$\omega$", fontsize=font_sizes["axis_labels"]
        )
    for row in range(len(n_agents_values)):
        axes[row, 0].set_ylabel(
            r"$\tau$", fontsize=font_sizes["axis_labels"]
        )
    return fig

In [ ]:
control_style = {"description_width": "initial"}
controls = {
    "observable": widgets.Dropdown(
        description="Observer",
        options=[
            (metadata["label"], observable)
            for observable, metadata in CONFIG["observables"].items()
        ],
        style=control_style,
        layout=widgets.Layout(width="320px"),
    ),
    "n_agents": widgets.Dropdown(
        description="n agents",
        options=[int(value) for value in results.coords["n_agents"].values],
        style=control_style,
    ),
    "composition": widgets.Dropdown(
        description=composition_dim.replace("_", " "),
        options=[
            (f"{value:g}", value)
            for value in results.coords[composition_dim].values.tolist()
        ],
        style=control_style,
    ),
}

plot_controls = widgets.HBox(
    list(controls.values()),
    layout=widgets.Layout(
        gap="12px",
        align_items="flex-start",
        flex_flow="row wrap",
    ),
)


def build_plot():
    figure = plot_last_step(
        controls["observable"].value,
        controls["n_agents"].value,
        controls["composition"].value,
    )
    plt.close(figure)
    return figure


def update_plot(_):
    try:
        plot_display.update(build_plot())
    except ValueError as error:
        plot_display.update(str(error))


display(plot_controls)
plot_display = display(build_plot(), display_id=True)
for control in controls.values():
    control.observe(update_plot, names="value")

## Visibility grid

Rows show population sizes and columns show sampled visibility radii. A shared color scale and population-specific Choi and maximum-ER references appear below the grid. The observer and composition controls above update this figure.

In [ ]:
def build_visibility_grid():
    figure = plot_visibility_grid(
        controls["observable"].value,
        controls["composition"].value,
    )
    plt.close(figure)
    return figure


def update_visibility_grid(_):
    try:
        visibility_grid_display.update(build_visibility_grid())
    except ValueError as error:
        visibility_grid_display.update(str(error))


visibility_grid_display = display(build_visibility_grid(), display_id=True)
for control_name in ("observable", "composition"):
    controls[control_name].observe(update_visibility_grid, names="value")

In [ ]:
run_id = Path(CONFIG["result_store"]).stem
figure_dir = CONFIG["export"]["output_root"] / run_id
figure_dir.mkdir(parents=True, exist_ok=True)

pdf_paths = []
for observable in CONFIG["observables"]:
    """
    for n_agents in results.coords["n_agents"].values:
        for composition in results.coords[composition_dim].values:
            figure = plot_last_step(
                observable, int(n_agents), composition.item()
            )
            filename = (
                f"{observable}__n_agents-{int(n_agents)}__"
                f"{composition_dim}-{composition:g}__{CONFIG['plot']['method']}.pdf"
            )
            pdf_path = figure_dir / filename
            figure.savefig(
                pdf_path,
                format="pdf",
                dpi=CONFIG["export"]["dpi"],
                bbox_inches="tight",
            )
            plt.close(figure)
            pdf_paths.append(pdf_path)
    """
    for composition in results.coords[composition_dim].values:
        figure = plot_visibility_grid(observable, composition.item())
        filename = (
            f"{observable}__{composition_dim}-{composition:g}__"
            f"visibility-grid__{CONFIG['plot']['method']}.pdf"
        )
        pdf_path = figure_dir / filename
        figure.savefig(
            pdf_path,
            format="pdf",
            dpi=CONFIG["export"]["dpi"],
            bbox_inches="tight",
        )
        plt.close(figure)
        pdf_paths.append(pdf_path)

print(
    f"Saved {len(pdf_paths)} PDFs to "
    f"{figure_dir.relative_to(project_root)}"
)